<a href="https://www.kaggle.com/code/jayhawk1900/f1-tabm?scriptVersionId=321465276" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# F1 Pit Stop Prediction — TabM (Parameter-Efficient Neural Ensemble)

**Competition:** [Playground Series S6E5 — Predicting F1 Pit Stops](https://www.kaggle.com/competitions/playground-series-s6e5)
**Metric:** ROC-AUC | **Task:** Binary classification — will this driver pit on the next lap?

---

## Overview

**TabM** is a tabular neural network (Yandex Research, 2024) built around **BatchEnsemble**: instead of training k independent networks, it uses one shared backbone weight matrix plus k lightweight per-member rank-1 adapters (two small vectors each). Each member scales the input and output by its own vectors, so they learn different functions and make different errors — a real ensemble at nearly single-model cost.

This is a deliberately **plain** TabM (no periodic numeric embeddings) to maximize error diversity from the RealMLP model, so the two blend well together. It uses the same feature-engineering pipeline and ORIG-concat strategy as the RealMLP notebook, with multi-seed averaging.

## Step 1: The BatchEnsemble Linear Layer

The single novel component. A standard linear layer has one weight matrix `W`. BatchEnsemble shares `W` across all `k` ensemble members but gives each member two cheap vectors:
- `r_i` (input scaling, length = in_features)
- `s_i` (output scaling, length = out_features)

Member `i` computes: `((x * r_i) @ W) * s_i + bias_i`

This is equivalent to each member having weight `W ⊙ (r_i ⊗ s_i)` — a per-member rank-1 modulation of the shared backbone. The bulk of the parameters (`W`) is shared; only the small `r`, `s`, and bias vary per member. Result: ensemble diversity at a fraction of the cost of `k` full models.

In [1]:
# ============================================================
# TabM — complete pipeline | plain (no periodic embeddings)
# BatchEnsemble neural ensemble, ORIG-concat, multi-seed
# ============================================================
import os, math, random, time, warnings
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder
import torch
import torch.nn as nn
warnings.filterwarnings('ignore')

def seed_everything(seed):
    np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

# ---------- Load data ----------
DATA_DIR = '/kaggle/input/competitions/playground-series-s6e5'
ORIG_DIR = '/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
orig = pd.read_csv(f'{ORIG_DIR}/f1_strategy_dataset_v4.csv')
print('Loaded:', train.shape, test.shape, orig.shape)

ID, TARGET = 'id', 'PitNextLap'
orig = orig.drop(['Normalized_TyreLife'], axis=1)
y_orig = orig[TARGET]; orig = orig.drop([TARGET], axis=1)
X = train.drop([ID, TARGET], axis=1); train_id = train[ID]
y = train[TARGET]
X_test = test.drop([ID], axis=1); test_id = test[ID]
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

# ---------- Feature engineering (same pipeline as RealMLP) ----------
category_map = {}
important_combos = [('Race', 'Compound'), ('Race', 'Year')]

def feature_engineering(df, fit=False):
    df = df.copy()
    df['_LapNumber_/_RaceProgress'] = (df['LapNumber'] / (df['RaceProgress'] + 1e-6)).astype('float32')
    df['_TyreLife_/_LapNumber'] = (df['TyreLife'] / df['LapNumber'].clip(lower=1)).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation'] = (df['LapTime (s)'] * df['Cumulative_Degradation']).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation_abs'] = (df['LapTime (s)'] * df['Cumulative_Degradation'].abs()).astype('float32')
    df['_LapTime (s)_/_Cumulative_Degradation_abs'] = (df['LapTime (s)'] / (df['Cumulative_Degradation'].abs() + 1e-6)).astype('float32')
    for col in cat_cols:
        if fit:
            codes, uniques = df[col].factorize(); category_map[col] = uniques
        else:
            uniques = category_map[col]; cm = {c: i for i, c in enumerate(uniques)}
            codes = df[col].map(cm).fillna(-1).astype('int32')
        df[col] = pd.Series(codes, index=df.index).astype('category')
    for col in num_cols + ['_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber']:
        cn = f"{col}_cat_" if col in num_cols else f"{col[1:]}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize(); category_map[col] = uniques
        else:
            uniques = category_map[col]; cm = {c: i for i, c in enumerate(uniques)}
            codes = np.floor(df[col]).map(cm).fillna(-1).astype('int32')
        df[cn] = pd.Series(codes, index=df.index).astype('category')
    for col in cat_cols + ['Year_cat_', 'PitStop_cat_']:
        nm = f"_{col}_count" if col in cat_cols else f"_{col[:-1]}_count"
        if fit:
            cmap = df[col].value_counts(); category_map[nm] = cmap
        else:
            cmap = category_map[nm]
        df[nm] = df[col].astype(object).map(cmap).fillna(0).astype('int32')
    for col, bins_list in {'RaceProgress': [200], 'LapTime (s)': [7]}.items():
        for nb in bins_list:
            bn = f"{col}_{nb}_quantile_bin_"
            if fit:
                kb = KBinsDiscretizer(n_bins=nb, encode='ordinal', strategy='quantile', subsample=None)
                b = kb.fit_transform(df[[col]]).ravel().astype('int32'); category_map[bn] = kb
            else:
                kb = category_map[bn]; b = kb.transform(df[[col]]).ravel().astype('int32')
            df[bn] = pd.Series(b, index=df.index).astype('category')
    combo_names = []
    for cols in important_combos:
        cn = '_'.join(cols) + '_'; combo_names.append(cn)
        s = df[cols[0]].astype(str)
        for c in cols[1:]: s = s + '_' + df[c].astype(str)
        if fit:
            codes, uniques = pd.factorize(s, sort=False); category_map[cn] = uniques
        else:
            uniques = category_map[cn]; cm = {c: i for i, c in enumerate(uniques)}
            codes = s.map(cm).fillna(-1).astype('int32')
        df[cn] = pd.Series(codes, index=df.index).astype('category')
    new_cat = [c for c in df.columns if c.endswith('_')]
    new_num = [c for c in df.columns if c.startswith('_')]
    return df, new_cat, new_num, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, _, _, _ = feature_engineering(X_test, fit=False)
orig, _, _, _ = feature_engineering(orig, fit=False)
cat_cols_all = cat_cols + new_cat_cols
print('After FE:', X.shape, '| cat:', len(cat_cols_all), '| combos:', combo_names)

# ---------- Components ----------
class NumPrep(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.med = np.median(X, axis=0)
        q = np.quantile(X,0.75,axis=0)-np.quantile(X,0.25,axis=0)
        z = q==0.0; q[z]=0.5*(X.max(axis=0)[z]-X.min(axis=0)[z])
        self.iqr = 1.0/(q+1e-30); self.iqr[q==0.0]=0.0
        return self
    def transform(self, X, y=None):
        X = X.copy().astype(np.float32)
        X = (X - self.med[None,:]) * self.iqr[None,:]
        X = X / np.sqrt(1 + (X/3)**2)   # smooth clip
        return X

class BatchEnsembleLinear(nn.Module):
    def __init__(self, in_f, out_f, k, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_f, out_f))
        self.r = nn.Parameter(torch.ones(k, in_f))
        self.s = nn.Parameter(torch.ones(k, out_f))
        self.bias = nn.Parameter(torch.zeros(k, out_f)) if bias else None
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        with torch.no_grad():
            self.r += 0.01*torch.randn_like(self.r)
            self.s += 0.01*torch.randn_like(self.s)
    def forward(self, x):
        x = x * self.r[None,:,:]
        x = torch.einsum('bki,io->bko', x, self.weight)
        x = x * self.s[None,:,:]
        if self.bias is not None: x = x + self.bias[None,:,:]
        return x

class TabM(nn.Module):
    def __init__(self, n_num, cat_dims, cfg):
        super().__init__()
        self.k = cfg['k']; ed = cfg['embed_dim']
        self.embeds = nn.ModuleList([nn.Embedding(d, ed) for d in cat_dims])
        in_dim = n_num + len(cat_dims)*ed
        hidden = cfg['hidden_dims']; drop = cfg['dropout']
        layers = []; d = in_dim
        for h in hidden:
            layers.append(BatchEnsembleLinear(d, h, self.k))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(drop))
            d = h
        self.backbone = nn.Sequential(*layers)
        self.head = BatchEnsembleLinear(d, 1, self.k)
    def forward(self, x_num, x_cat):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeds)]
        x = torch.cat([x_num] + embs, dim=1)            # (batch, in_dim)
        x = x.unsqueeze(1).expand(-1, self.k, -1)        # (batch, k, in_dim)
        x = self.backbone(x)
        x = self.head(x)                                 # (batch, k, 1)
        return torch.sigmoid(x)

class TabMClassifier(BaseEstimator):
    def __init__(self, **kw): self.params = {**CONFIG, **kw}
    def fit(self, X_train, y_train, X_val, y_val, cat_col_names=None, ckpt='tabm.pth', X_test=None):
        p = self.params; dev = device
        cat_col_names = cat_col_names or []
        num_cols_ = [c for c in X_train.columns if c not in cat_col_names]
        self.num_cols_ = num_cols_; self.cat_cols_ = cat_col_names
        Xtn = X_train[num_cols_].values.astype(np.float32)
        Xvn = X_val[num_cols_].values.astype(np.float32)
        Xtc = X_train[cat_col_names].values.astype(np.int64)
        Xvc = X_val[cat_col_names].values.astype(np.int64)
        y_tr = np.asarray(y_train); y_v = np.asarray(y_val)
        self.prep = NumPrep().fit(Xtn)
        Xtn = self.prep.transform(Xtn); Xvn = self.prep.transform(Xvn)
        allc = [Xtc, Xvc]
        if X_test is not None: allc.append(X_test[cat_col_names].values.astype(np.int64))
        cat_dims = (np.concatenate(allc, axis=0).max(axis=0)+1).tolist()
        self.cat_dims_ = cat_dims
        cmax = np.array(cat_dims)-1
        Xtc = np.clip(Xtc, 0, cmax); Xvc = np.clip(Xvc, 0, cmax)
        w = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
        pos_w = torch.tensor(w[1], dtype=torch.float32, device=dev)
        self.classes_ = np.unique(y_tr)
        self.model = TabM(Xtn.shape[1], cat_dims, p).to(dev)
        opt = torch.optim.AdamW(self.model.parameters(), lr=p['lr'], weight_decay=p['weight_decay'])
        Xtn_t = torch.as_tensor(Xtn, device=dev)
        Xtc_t = torch.as_tensor(Xtc, dtype=torch.long, device=dev)
        ytt = torch.as_tensor(y_tr, dtype=torch.float32, device=dev)
        Xvn_t = torch.as_tensor(Xvn, device=dev)
        Xvc_t = torch.as_tensor(Xvc, dtype=torch.long, device=dev)
        k = p['k']; bs = p['train_bs']; ebs = p['eval_bs']; epochs = p['epochs']
        total = epochs * len(y_tr); order = np.arange(len(y_tr))
        best = -np.inf; best_vp = None; step = 0
        for epoch in range(epochs):
            self.model.train()
            for st in range(0, len(y_tr), bs):
                idx = order[st:st+bs]
                lr = p['lr'] * (math.cos(math.pi * step / total) + 1) / 2  # cosine decay
                for g in opt.param_groups: g['lr'] = lr
                step += 1
                opt.zero_grad()
                out = self.model(Xtn_t[idx], Xtc_t[idx]).squeeze(-1)  # (b, k)
                yb = ytt[idx].unsqueeze(1).expand(-1, k)              # (b, k)
                yp = out.clamp(1e-7, 1-1e-7)
                loss = (-pos_w*yb*torch.log(yp) - (1-yb)*torch.log(1-yp)).mean()
                loss.backward(); opt.step()
            np.random.shuffle(order)
            self.model.eval()
            with torch.no_grad():
                vp = np.concatenate([self.model(Xvn_t[s:s+ebs], Xvc_t[s:s+ebs]).mean(dim=1).squeeze(-1).cpu().numpy()
                                     for s in range(0, len(y_v), ebs)])
            sc = roc_auc_score(y_v, vp)
            if sc > best: best = sc; best_vp = vp.copy(); torch.save(self.model.state_dict(), ckpt)
            if p['verbosity'] >= 2: print(f"  epoch {epoch+1}/{epochs}  score={sc:.5f}  best={best:.5f}")
        self.model.load_state_dict(torch.load(ckpt))
        self.best_score_ = best; self.best_val_probs_ = best_vp
        return self
    def predict_proba(self, X):
        ebs = self.params['eval_bs']
        Xn = self.prep.transform(X[self.num_cols_].values.astype(np.float32))
        Xc = np.clip(X[self.cat_cols_].values.astype(np.int64), 0, np.array(self.cat_dims_)-1)
        Xn = torch.as_tensor(Xn, device=device)
        Xc = torch.as_tensor(Xc, dtype=torch.long, device=device)
        self.model.eval()
        with torch.no_grad():
            pp = np.concatenate([self.model(Xn[s:s+ebs], Xc[s:s+ebs]).mean(dim=1).squeeze(-1).cpu().numpy()
                                 for s in range(0, len(Xn), ebs)])
        return np.stack([1-pp, pp], axis=1)

# ---------- CONFIG ----------
CONFIG = {
    'k': 32, 'embed_dim': 8, 'hidden_dims': [256, 256, 128], 'dropout': 0.1,
    'lr': 0.002, 'weight_decay': 0.0005, 'epochs': 6,
    'train_bs': 512, 'eval_bs': 16384, 'verbosity': 2,
}
print('CONFIG ready. k =', CONFIG['k'], '| epochs =', CONFIG['epochs'])

# ---------- Multi-seed K-fold ----------
FOLDS = 5; SEEDS = [42, 123, 7]
oof_preds = np.zeros(len(X)); test_preds = np.zeros(len(X_test)); seed_oofs = []
t_all = time.time()
for si, S in enumerate(SEEDS):
    print(f'\n{"="*40}\n=== SEED {S} ({si+1}/{len(SEEDS)}) ===\n{"="*40}')
    seed_everything(S)
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=S)
    oof_seed = np.zeros(len(X)); test_seed = np.zeros(len(X_test))
    for fold, ((tr, va), (otr, _)) in enumerate(zip(skf.split(X, y), skf.split(orig, y_orig)), 1):
        X_tr = pd.concat([X.iloc[tr], orig.iloc[otr]], axis=0).reset_index(drop=True)
        y_tr = pd.concat([y.iloc[tr], y_orig.iloc[otr]], axis=0).reset_index(drop=True)
        X_val = X.iloc[va].copy(); y_val = y.iloc[va]; X_tst = X_test.copy()
        te = TargetEncoder(cv=FOLDS, smooth='auto', shuffle=True, random_state=S)
        X_tr_te = te.fit_transform(X_tr[combo_names], y_tr)
        X_val_te = te.transform(X_val[combo_names]); X_tst_te = te.transform(X_tst[combo_names])
        ten = [f"_{c}TE" for c in combo_names]
        X_tr[ten] = X_tr_te; X_val[ten] = X_val_te; X_tst[ten] = X_tst_te
        if si == 0 and fold == 1: print('Total features:', len(X_tr.columns))
        m = TabMClassifier(**CONFIG)
        m.fit(X_tr, y_tr, X_val, y_val, cat_col_names=cat_cols_all, ckpt=f't_s{S}_f{fold}.pth')
        oof_seed[va] = m.best_val_probs_
        test_seed += m.predict_proba(X_tst)[:, 1] / FOLDS
        print(f'  Seed {S} Fold {fold}: {roc_auc_score(y_val, m.best_val_probs_):.5f}')
        torch.cuda.empty_cache()
    sa = roc_auc_score(y, oof_seed); seed_oofs.append(sa)
    print(f'>>> Seed {S} OOF: {sa:.5f}')
    oof_preds += oof_seed / len(SEEDS); test_preds += test_seed / len(SEEDS)

oof_auc = roc_auc_score(y, oof_preds)
print(f'\n{"="*40}')
print(f'Per-seed OOFs: {[round(s,5) for s in seed_oofs]}')
print(f'TabM multi-seed OOF: {oof_auc:.5f}')
print(f'(RealMLP multi-seed was: 0.95409)')
print(f'Total time: {(time.time()-t_all)/60:.1f} min')
print("="*40)

np.save('/kaggle/working/oof_tabm.npy', oof_preds)
np.save('/kaggle/working/pred_tabm.npy', test_preds)
sub = pd.DataFrame({ID: test_id, TARGET: test_preds})
sub.to_csv('/kaggle/working/submission.csv', index=False)
print('Saved oof_tabm.npy, pred_tabm.npy, submission.csv')

PyTorch: 2.10.0+cu128 | device: cuda
Loaded: (439140, 16) (188165, 15) (101371, 16)
After FE: (439140, 41) | cat: 20 | combos: ['Race_Compound_', 'Race_Year_']
CONFIG ready. k = 32 | epochs = 6

=== SEED 42 (1/3) ===
Total features: 43
  epoch 1/6  score=0.90900  best=0.90900
  epoch 2/6  score=0.94709  best=0.94709
  epoch 3/6  score=0.94870  best=0.94870
  epoch 4/6  score=0.94795  best=0.94870
  epoch 5/6  score=0.94843  best=0.94870
  epoch 6/6  score=0.94818  best=0.94870
  Seed 42 Fold 1: 0.94870
  epoch 1/6  score=0.90475  best=0.90475
  epoch 2/6  score=0.94457  best=0.94457
  epoch 3/6  score=0.94581  best=0.94581
  epoch 4/6  score=0.94633  best=0.94633
  epoch 5/6  score=0.94617  best=0.94633
  epoch 6/6  score=0.94521  best=0.94633
  Seed 42 Fold 2: 0.94633
  epoch 1/6  score=0.90648  best=0.90648
  epoch 2/6  score=0.94590  best=0.94590
  epoch 3/6  score=0.94722  best=0.94722
  epoch 4/6  score=0.94768  best=0.94768
  epoch 5/6  score=0.94680  best=0.94768
  epoch 6/6  sc